# 03 — Positive control: ABC vs GCB overall survival (GSE181063 training set only)

**Goal.** Reproduce a well-established finding before building any model: in R-CHOP-treated DLBCL, the activated B-cell (ABC) subtype has worse overall survival than the germinal-centre B-cell (GCB) subtype. If our cleaned data and survival coding are correct, we should see this in the training set.

**Leakage rules for this notebook**
- Only the 380 training patients are used. The test file `expr_GSE181063_test.parquet` is never opened, and test rows are dropped from memory right after loading.
- The hypothesis, contrast and success criterion are written to a plan file **before** any survival comparison is run.
- Nothing learned here is used to choose genes for step 4.

TR: Model kurmadan önce bilinen bir bulguyu (ABC'nin GCB'ye göre daha kötü OS'si) yalnızca eğitim setinde yeniden buluyoruz. Test seti açılmıyor; hipotez ve başarı ölçütü analizden önce dosyaya yazılıyor.

## 1. Setup
TR: `lifelines` paketini kurar, Drive'ı bağlar ve klasör yollarını tanımlar.

In [ ]:
!pip -q install lifelines

In [ ]:
import os, json, hashlib, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test
from lifelines.plotting import add_at_risk_counts
from lifelines.utils import median_survival_times
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/dlbcl-prognosis'
CLEAN = f'{PROJECT_DIR}/data/clean'
PROC  = f'{PROJECT_DIR}/data/processed'
RES   = f'{PROJECT_DIR}/results/03'
FIG   = f'{PROJECT_DIR}/figures/03'
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True)

HORIZON = 60          # months; 5-year OS is reported
GROUPS = ['ABC', 'GCB']

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

print('ready')

## 2. Pre-specify the analysis (before looking at any survival comparison)
The plan is saved to `results/03/plan_03.json`. If the file already exists (re-run), the notebook checks that the plan has not changed; it refuses to continue if it has. This is a small-scale version of the pre-registration we will do in step 4.

TR: Hipotezi, karşılaştırmayı ve başarı ölçütünü analizden önce kaydeder. Not defterini tekrar çalıştırırsan planın değişmediğini kontrol eder; değiştiyse durur. Sonuca bakıp planı değiştirmek "p-hacking" olur.

In [ ]:
plan = {
    'step': '03_positive_control',
    'dataset': 'GSE181063, training split only (n=380)',
    'endpoint': 'overall survival (os_time_months, os_event; 1 = death)',
    'primary_contrast': 'ABC vs GCB (GCB = reference); MHG and UNC excluded from the primary contrast, shown descriptively',
    'primary_tests': ['Kaplan-Meier', 'two-sided log-rank test', 'univariable Cox HR with 95% CI'],
    'expected_direction': 'HR(ABC vs GCB) > 1',
    'success_criterion': 'HR > 1 with 95% CI excluding 1 and log-rank p < 0.05',
    'if_not_met': 'investigate coding/linkage errors; do NOT change cohort, endpoint or contrast to obtain significance',
    'secondary': ['proportional-hazards check (Schoenfeld residual test)',
                  'Cox ABC vs GCB adjusted for IPI (complete cases), compared with unadjusted HR in the same subset',
                  'sensitivity: exclude sex_mismatch_confident samples',
                  'sensitivity: exclude correlation outliers',
                  'label consistency: Lymph2Cx-gene COO score vs provided COO labels (no survival data used)'],
}
plan_path = f'{RES}/plan_03.json'
if os.path.exists(plan_path):
    with open(plan_path) as f:
        saved = json.load(f)
    assert {k: v for k, v in saved.items() if k != 'saved_at'} == plan, \
        'Plan differs from the saved version. Do not edit the plan after seeing results.'
    print('Plan unchanged; saved at', saved['saved_at'])
else:
    plan['saved_at'] = datetime.datetime.now().isoformat(timespec='seconds')
    with open(plan_path, 'w') as f:
        json.dump(plan, f, indent=2)
    plan.pop('saved_at')
    print('Plan saved:', plan_path)

## 3. Load the training set only
Test rows are removed immediately and the full clinical table is deleted from memory, so later cells cannot touch them by accident.

TR: Klinik tabloyu ve kilitli bölmeyi okur, yalnızca 380 eğitim hastasını tutar. Hasta ve ölüm sayılarını 2a ile karşılaştırır (380 / 173). Test satırları hemen bellekten silinir.

In [ ]:
clin  = pd.read_csv(f'{CLEAN}/clinical_clean_GSE181063.csv')
split = pd.read_csv(f'{CLEAN}/split_GSE181063.csv')

train_ids = set(split.loc[split['set'] == 'train', 'sample_id'])
test_ids  = set(split.loc[split['set'] == 'test',  'sample_id'])
assert train_ids.isdisjoint(test_ids)

train = clin[clin['sample_id'].isin(train_ids)].copy()
del clin   # test rows are no longer in memory

assert len(train) == 380, len(train)
assert int(train['os_event'].sum()) == 173, int(train['os_event'].sum())
assert not train['sample_id'].isin(test_ids).any()
assert train['os_event'].isin([0, 1]).all() and (train['os_time_months'] > 0).all()

print(f"Training patients: {len(train)} | deaths: {int(train['os_event'].sum())}")
print('\nCOO labels (counts and deaths):')
print(train.groupby('coo', dropna=False)['os_event'].agg(patients='size', deaths='sum'))

## 4. Kaplan-Meier: ABC vs GCB (primary) and all four groups (descriptive)
TR: ABC ve GCB için Kaplan-Meier eğrilerini, risk altındaki hasta sayılarını, medyan OS'yi ve 5 yıllık OS'yi (%95 GA) hesaplar. Dört grubun (MHG ve UNC dahil) grafiği sadece betimleyici.

In [ ]:
def km_summary(df, groups, horizon=HORIZON):
    rows, fits = [], {}
    for g in groups:
        d = df[df['coo'] == g]
        kmf = KaplanMeierFitter().fit(d['os_time_months'], d['os_event'], label=g)
        fits[g] = kmf
        s = kmf.survival_function_.loc[:horizon].iloc[-1, 0]
        ci = kmf.confidence_interval_.loc[:horizon].iloc[-1]
        med = kmf.median_survival_time_
        med_ci = median_survival_times(kmf.confidence_interval_).iloc[0]
        rows.append({'group': g, 'patients': len(d), 'deaths': int(d['os_event'].sum()),
                     'median_OS_months': med, 'median_CI_low': med_ci.iloc[0], 'median_CI_high': med_ci.iloc[1],
                     f'OS_{horizon}m': s, f'OS_{horizon}m_CI_low': ci.iloc[0], f'OS_{horizon}m_CI_high': ci.iloc[1]})
    return pd.DataFrame(rows), fits

def km_plot(fits, title, path):
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for kmf in fits.values():
        kmf.plot_survival_function(ax=ax, ci_show=True, show_censors=True, censor_styles={'ms': 4})
    ax.set_xlabel('Months from diagnosis'); ax.set_ylabel('Overall survival probability')
    ax.set_ylim(0, 1.02); ax.set_title(title)
    add_at_risk_counts(*fits.values(), ax=ax, rows_to_show=['At risk'])
    plt.tight_layout(); fig.savefig(path, dpi=300, bbox_inches='tight'); plt.show()

ab = train[train['coo'].isin(GROUPS)].copy()
km_tab, km_fits = km_summary(ab, GROUPS)
display(km_tab.round(3))
km_plot(km_fits, 'GSE181063 training set: OS by COO (ABC vs GCB)', f'{FIG}/km_ABC_vs_GCB.png')

all_groups = [g for g in ['ABC', 'GCB', 'MHG', 'UNC'] if (train['coo'] == g).any()]
km_tab_all, km_fits_all = km_summary(train, all_groups)
display(km_tab_all.round(3))
km_plot(km_fits_all, 'GSE181063 training set: OS by all COO groups (descriptive)', f'{FIG}/km_all_COO.png')

## 5. Log-rank test
TR: ABC ile GCB arasında iki yönlü log-rank testi (birincil). Dört grup için genel log-rank yalnızca betimleyici.

In [ ]:
a, g = ab[ab['coo'] == 'ABC'], ab[ab['coo'] == 'GCB']
lr = logrank_test(a['os_time_months'], g['os_time_months'], a['os_event'], g['os_event'])
print(f'Log-rank ABC vs GCB: chi2 = {lr.test_statistic:.2f}, p = {lr.p_value:.2e}')

lr_all = multivariate_logrank_test(train['os_time_months'], train['coo'], train['os_event'])
print(f'Log-rank across {len(all_groups)} groups (descriptive): chi2 = {lr_all.test_statistic:.2f}, p = {lr_all.p_value:.2e}')

## 6. Cox proportional hazards: hazard ratio ABC vs GCB
`abc = 1` for ABC, `0` for GCB (reference). HR > 1 means higher risk of death for ABC. We also test the proportional-hazards assumption: a small p-value means the HR is not constant over time, so a single HR summarises the effect less well (it does not make the result invalid, but we must report it).

TR: Tek değişkenli Cox modeli ile ABC'nin GCB'ye göre ölüm riski oranını (HR, %95 GA) hesaplar. Ardından orantılı riskler varsayımını Schoenfeld artıklarıyla test eder.

In [ ]:
ab['abc'] = (ab['coo'] == 'ABC').astype(int)

def cox_abc(df, covars=('abc',), label=''):
    cph = CoxPHFitter()
    cph.fit(df[['os_time_months', 'os_event', *covars]], 'os_time_months', 'os_event')
    s = cph.summary.loc['abc']
    return cph, {'model': label, 'n': len(df), 'deaths': int(df['os_event'].sum()),
                 'HR_ABC_vs_GCB': s['exp(coef)'], 'CI_low': s['exp(coef) lower 95%'],
                 'CI_high': s['exp(coef) upper 95%'], 'p': s['p'], 'C_index': cph.concordance_index_}

cox_rows = []
cph_uni, row = cox_abc(ab, label='primary: univariable')
cox_rows.append(row)
print(pd.DataFrame([row]).round(4).to_string(index=False))

ph = proportional_hazard_test(cph_uni, ab[['os_time_months', 'os_event', 'abc']], time_transform='rank')
print(f"\nProportional-hazards test (rank time): p = {ph.summary.loc['abc', 'p']:.3f}")

## 7. Secondary: is the COO effect independent of IPI?
IPI is missing for some patients, so the adjusted model uses complete cases. To compare fairly, the unadjusted HR is recomputed in the **same** subset. If the HR shrinks a lot after adjustment, part of the ABC effect is explained by ABC patients having higher IPI. This is exploratory; the formal comparison with IPI is step 7.

TR: COO etkisinin klinik risk skorundan (IPI) bağımsız olup olmadığına bakar. Adil kıyas için düzeltmesiz HR'yi de aynı eksiksiz alt kümede yeniden hesaplar. Bu ikincil ve keşif amaçlı; asıl kıyas 7. adımda.

In [ ]:
cc = ab.dropna(subset=['ipi']).copy()
print(f'Complete cases with IPI: {len(cc)}/{len(ab)}')
_, row = cox_abc(cc, label='secondary: unadjusted, IPI complete cases'); cox_rows.append(row)
cph_adj, row = cox_abc(cc, covars=('abc', 'ipi'), label='secondary: adjusted for IPI (0-5, linear)'); cox_rows.append(row)
display(cph_adj.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].round(4))

## 8. Sensitivity analyses using the step 2b QC flags
Pre-specified in step 2b: (a) exclude samples with a confident sex mismatch (clinical data may belong to another sample), (b) exclude correlation outliers. The column names are detected from `qc_samples.csv`; if detection fails, the cell prints the available columns so we can fix it.

TR: 2b'deki QC işaretleriyle iki duyarlılık analizi: kesin cinsiyet uyuşmazlığı olan örnekleri, sonra korelasyon aykırılarını çıkarıp HR'yi yeniden hesaplar. Sonuç büyük ölçüde değişmemeli.

In [ ]:
qc = pd.read_csv(f'{PROC}/qc_samples.csv')
print('qc_samples.csv columns:', qc.columns.tolist())
id_col = 'sample_id' if 'sample_id' in qc.columns else qc.columns[0]

def as_bool(s):
    return s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'yes'])

def find_col(keyword):
    hits = [c for c in qc.columns if keyword in c.lower()]
    assert len(hits) == 1, f'expected one column containing "{keyword}", found {hits}'
    return hits[0]

for key, label in [('confident', 'sensitivity: excl. confident sex mismatch'),
                   ('outlier', 'sensitivity: excl. correlation outliers')]:
    col = find_col(key)
    flagged = set(qc.loc[as_bool(qc[col]), id_col])
    sub = ab[~ab['sample_id'].isin(flagged)]
    print(f'{col}: {len(ab) - len(sub)} ABC/GCB training samples excluded')
    _, row = cox_abc(sub, label=label); cox_rows.append(row)

cox_tab = pd.DataFrame(cox_rows)
display(cox_tab.round(4))

## 9. Label consistency check (no survival data)
Do the provided COO labels agree with the expression data of the same samples? We compute a simple score from the published Lymph2Cx genes (Scott et al., 2014): mean z-score of ABC genes minus mean z-score of GCB genes, using **training samples only**. ABC-labelled samples should score higher.

Note: a very high AUC here is **expected** and not a sign of error (rule 10 applies to survival models). The labels in GSE181063 were themselves derived from gene expression on these arrays, so this is a check of sample–label linkage, not an independent validation of COO. Some genes may be missing (GPL14951 coverage or the step 2b low-expression filter); missing genes are listed.

TR: Verilen COO etiketleri, aynı örneklerin ifade verisiyle uyumlu mu? Lymph2Cx genlerinden basit bir skor hesaplar (yalnızca eğitim örnekleri). Burada yüksek AUC beklenir, çünkü etiketler zaten bu ifade verisinden türetildi; bu bir örnek–etiket eşleşmesi kontrolü, bağımsız doğrulama değil.

In [ ]:
expr = pd.read_parquet(f'{PROC}/expr_GSE181063_train.parquet')
if not (set(expr.columns) & train_ids):
    expr = expr.T                      # make it genes x samples
expr = expr[[c for c in expr.columns if c in train_ids]]
assert expr.shape[1] == 380 and not (set(expr.columns) & test_ids)
expr.index = expr.index.astype(str)
print('Expression (train):', expr.shape)

gt = pd.read_csv(f'{PROC}/gene_table_common.csv', dtype=str)
print('gene_table_common.csv columns:', gt.columns.tolist())
key_col = max(gt.columns, key=lambda c: gt[c].isin(expr.index).mean())
sym_cols = [c for c in gt.columns if 'symbol' in c.lower()]
assert gt[key_col].isin(expr.index).mean() > 0.9 and sym_cols, 'could not match gene table to expression index'
sym2key = {}
for c in sym_cols:
    for s, k in zip(gt[c], gt[key_col]):
        if isinstance(s, str):
            sym2key.setdefault(s.upper(), k)
for k in expr.index:                    # if the index itself is symbols
    sym2key.setdefault(k.upper(), k)

ABC_GENES = ['TNFRSF13B', 'LIMD1', 'IRF4', 'CREB3L2', 'PIM2', 'CYB5R2', 'RAB7L1|RAB29', 'CCDC50']
GCB_GENES = ['MME', 'SERPINA9', 'ASB13', 'MAML3', 'ITPKB', 'MYBL1', 'S1PR2']

def resolve(genes):
    found, missing = {}, []
    for g in genes:
        keys = [sym2key[a] for a in g.split('|') if a in sym2key]
        (found.__setitem__(g, keys[0]) if keys else missing.append(g))
    return found, missing

abc_f, abc_m = resolve(ABC_GENES); gcb_f, gcb_m = resolve(GCB_GENES)
print(f'ABC genes found {len(abc_f)}/{len(ABC_GENES)}, missing: {abc_m}')
print(f'GCB genes found {len(gcb_f)}/{len(GCB_GENES)}, missing: {gcb_m}')

z = expr.sub(expr.mean(axis=1), axis=0).div(expr.std(axis=1, ddof=1), axis=0)   # training-only z-scores
score = z.loc[list(abc_f.values())].mean() - z.loc[list(gcb_f.values())].mean()
sc = train.set_index('sample_id')[['coo']].join(score.rename('coo_score'))

print('\nMedian score by label:'); print(sc.groupby('coo')['coo_score'].median().round(2))
s_ab = sc[sc['coo'].isin(GROUPS)]
auc = roc_auc_score((s_ab['coo'] == 'ABC').astype(int), s_ab['coo_score'])
mw = mannwhitneyu(s_ab.loc[s_ab.coo == 'ABC', 'coo_score'], s_ab.loc[s_ab.coo == 'GCB', 'coo_score'])
print(f'\nABC vs GCB: AUC = {auc:.3f}, Mann-Whitney p = {mw.pvalue:.1e}')

order = [g for g in ['ABC', 'GCB', 'MHG', 'UNC'] if (sc['coo'] == g).any()]
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.boxplot([sc.loc[sc.coo == g, 'coo_score'].dropna() for g in order])
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order)
ax.set_ylabel('Lymph2Cx-gene score (ABC − GCB, z)'); ax.set_title('COO label vs expression score (training set)')
plt.tight_layout(); fig.savefig(f'{FIG}/coo_score_by_label.png', dpi=300); plt.show()

## 10. Save results and manifest
TR: Tabloları `results/03/` içine, grafikleri `figures/03/` içine kaydeder; girdi dosyalarının SHA-256 değerlerini manifest'e yazar.

In [ ]:
km_tab.to_csv(f'{RES}/km_summary_ABC_GCB.csv', index=False)
km_tab_all.to_csv(f'{RES}/km_summary_all_COO.csv', index=False)
cox_tab.to_csv(f'{RES}/cox_results.csv', index=False)
pd.DataFrame([{'test': 'log-rank ABC vs GCB', 'chi2': lr.test_statistic, 'p': lr.p_value},
              {'test': f'log-rank {len(all_groups)} groups (descriptive)', 'chi2': lr_all.test_statistic, 'p': lr_all.p_value}]
             ).to_csv(f'{RES}/logrank_results.csv', index=False)

prim = cox_tab.iloc[0]
met = bool(prim['HR_ABC_vs_GCB'] > 1 and prim['CI_low'] > 1 and lr.p_value < 0.05)
inputs = [f'{CLEAN}/clinical_clean_GSE181063.csv', f'{CLEAN}/split_GSE181063.csv',
          f'{PROC}/qc_samples.csv', f'{PROC}/expr_GSE181063_train.parquet', f'{PROC}/gene_table_common.csv']
manifest = {
    'notebook': '03_positive_control.ipynb',
    'run_at': datetime.datetime.now().isoformat(timespec='seconds'),
    'inputs_sha256': {os.path.basename(p): sha256(p) for p in inputs},
    'test_set_opened': False,
    'n_train_ABC_GCB': int(len(ab)),
    'primary': {'HR': float(prim['HR_ABC_vs_GCB']), 'CI': [float(prim['CI_low']), float(prim['CI_high'])],
                'logrank_p': float(lr.p_value), 'PH_test_p': float(ph.summary.loc['abc', 'p'])},
    'success_criterion_met': met,
    'coo_score_auc': float(auc),
}
with open(f'{RES}/manifest_03.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest['primary'], indent=2))
print('Success criterion met:', met)
print('\nDone. Close this Colab session (Runtime > Disconnect and delete runtime).')